# PART 1: Text Processing and Exploratory Data Analysis 

## Task 1. Data preparation

In [1]:
import json, re
import pandas as pd #to see the data in a table format
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
DATA_FILE = "../../data/fashion_products_dataset.json"

def load_docs(path):
    with open(path, encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        return json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]

def load_docs(path):
    with open(path, encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        return json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]

docs = load_docs(DATA_FILE)
print(len(docs), "documents")
docs[0]    


FileNotFoundError: [Errno 2] No such file or directory: '../../data/fashion_products_dataset.json'

1.1 As a first step, you must pre-process the documents. In particular, for the text fields (title,
description) you should:

    - Removing stop words
    - Tokenization
    - Removing punctuation marks
    - Stemming
    - and... anything else you think it's needed (bonus point)

In [ ]:
# Remove stopwords
def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]


def tokenize(text):
    return word_tokenize(text)   # or text.split() once punctuation is removed

def remove_punctuation(text):
    # replace with a space so "look.Designed" becomes two tokens
    return re.sub(r"[^a-z0-9\s]", " ", text)

def stem(tokens):
    return [STEMMER.stem(t) for t in tokens]

def normalize(text):
    text = text.lower()                        # case folding
    text = re.sub(r"https?://\S+", " ", text) # remove URLs
    return text

def drop_short(tokens):
    return [t for t in tokens if len(t) > 1 or t.isdigit()]  # drop stray single letters

# Final function calling all of the above
def build_terms(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = remove_punctuation(normalize(text))
    tokens = text.split()
    tokens = remove_stopwords(tokens)
    tokens = drop_short(tokens)
    return stem(tokens)

#see changes in first document
text = docs[0]["description"]
print(text)
print(build_terms(text))

Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India
['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']


1.2 Take into account that for future queries, the final output must return (when present) the
following information for each of the selected documents:
pid, title, description, brand, category, sub_category, product_details, seller, out_of_stock,
selling_price, discount, actual_price, average_rating, url


In [ ]:
OUTPUT_FIELDS = ["pid", "title", "description", "brand", "category", "sub_category",
                 "product_details", "seller", "out_of_stock", "selling_price",
                 "discount", "actual_price", "average_rating", "url"]

def keep_output_fields(doc):
    return {f: doc.get(f) for f in OUTPUT_FIELDS}   # missing fields become None

1.3 Decide how to handle the fields category, sub_category, brand, product_details, and seller
during pre-processing. Should they be merged into a single text field, indexed as separate fields
in the inverted index or any other alternative? Justify your choice, considering how their
distinctiveness may affect retrieval effectiveness. What are pros and cons of each approach?

In [ ]:
def flatten_details(details):
    flat = {}
    for item in details or []:
        if isinstance(item, dict):
            flat.update({str(k): str(v) for k, v in item.items()})
    return flat

def details_text(flat):
    skip = {"style code", "sku", "model number", "pack of"}
    return " ".join(f"{k} {v}" for k, v in flat.items() if k.lower() not in skip)

def build_field_terms(doc):
    flat = flatten_details(doc.get("product_details"))
    fields = {
        "title_terms":       build_terms(doc.get("title")),
        "description_terms": build_terms(doc.get("description")),
        "brand_terms":       build_terms(doc.get("brand")),
        "category_terms":    build_terms(f"{doc.get('category') or ''} {doc.get('sub_category') or ''}"),
        "details_terms":     build_terms(details_text(flat)),
        "seller_terms":      build_terms(doc.get("seller")),
    }
    # optional combined field (the "merged" alternative, for comparison in the report)
    fields["all_terms"] = (fields["title_terms"] + fields["description_terms"] + fields["brand_terms"]
                           + fields["category_terms"] + fields["details_terms"])
    return fields

fields = build_field_terms(docs[0])
for name, terms in fields.items():
    print(name, terms)

title_terms ['solid', 'women', 'multicolor', 'track', 'pant']
description_terms ['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']
brand_terms ['york']
category_terms ['cloth', 'accessori', 'bottomwear']
details_terms ['closur', 'elast', 'pocket', 'side', 'pocket', 'fabric', 'cotton', 'blend', 'pattern', 'solid', 'color', 'multicolor']
seller_terms ['shyam', 'enterpris']
all_terms ['solid', 'women', 'multicolor', 'track', 'pant', 'yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india', 'york', 'cloth', 'accessori', 'bottomwear', 'closur', 'elast', 'pocket', 'side', 'pocket', 'fabric', 'cotton', 'blend', 'pattern', 'solid', 'color', 'multicol

1.4 Consider the fields out_of_stock, selling_price, discount, actual_price, and average_rating.
Decide how these should be handled during pre-processing to use in further search. Should
they be indexed as textual terms?


In [ ]:
def to_float(value):
    """'2,999' -> 2999.0 ; '69% off' -> 69.0 ; '' / None -> None"""
    if value is None:
        return None
    m = re.search(r"\d[\d,]*\.?\d*", str(value))
    return float(m.group().replace(",", "")) if m else None

def process_numeric(doc):
    return {
        "out_of_stock":   doc.get("out_of_stock"),            # keep as boolean
        "selling_price":  to_float(doc.get("selling_price")),
        "actual_price":   to_float(doc.get("actual_price")),
        "discount":       to_float(doc.get("discount")),
        "average_rating": to_float(doc.get("average_rating")),
    }

In [ ]:
def process_doc(doc):
    out = keep_output_fields(doc)
    out.update(process_numeric(doc))   # overwrites with typed values
    out.update(build_field_terms(doc))
    return out

if __name__ == "__main__":
    docs = load_docs(DATA_FILE)
    seen, processed = set(), []
    for d in docs:
        if d.get("pid") in seen:
            continue
        seen.add(d.get("pid"))
        processed.append(process_doc(d))
    with open("processed_docs.json", "w", encoding="utf-8") as f:
        json.dump(processed, f, ensure_ascii=False)

## Task 2. Exploratory Data Analysis

In [ ]:
OUTPUT_FIELDS = ["pid", "title", "description", "brand", "category", "sub_category",
                 "product_details", "seller", "out_of_stock", "selling_price",
                 "discount", "actual_price", "average_rating", "url"]

def keep_output_fields(doc):
    """Return only the fields the search output must show.
    Fields missing in the raw document are kept as None ("when present")."""
    return {f: doc.get(f) for f in OUTPUT_FIELDS}

In [ ]:
doc_store = {}
for d in docs:
    pid = d.get("pid")
    if pid is not None and pid not in doc_store:   # skip duplicates
        doc_store[pid] = keep_output_fields(d)

print(len(doc_store), "documents in the store")
doc_store[docs[0]["pid"]]   # check one entry

28080 documents in the store


{'pid': 'TKPFCZ9EA7H5FYZH',
 'title': 'Solid Women Multicolor Track Pants',
 'description': 'Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India',
 'brand': 'York',
 'category': 'Clothing and Accessories',
 'sub_category': 'Bottomwear',
 'product_details': [{'Style Code': '1005COMBO2'},
  {'Closure': 'Elastic'},
  {'Pockets': 'Side Pockets'},
  {'Fabric': 'Cotton Blend'},
  {'Pattern': 'Solid'},
  {'Color': 'Multicolor'}],
 'seller': 'Shyam Enterprises',
 'out_of_stock': False,
 'selling_price': '921',
 'discount': '69% off',
 'actual_price': '2,999',
 'average_rating': '3.9',
 'url': 'https://www.flipkart.com/yorker-solid-men-multicolor-track-pants/p/itmd2c76aadce459?pid=TKPFCZ9EA7H5FYZH&lid=LSTTKPFCZ9EA7H5FYZHVYXWP0&marketplace=FLIPKART&srno=b_1_1&otracker=browse&fm=organic&iid=177a46eb-d053-4732-b3de-fcad6ff59cbd.TKPFCZ9EA7H5FYZH.SEARCH&ssid=utkd4t

In [ ]:
# 1. every stored document has exactly the required fields
assert all(set(v.keys()) == set(OUTPUT_FIELDS) for v in doc_store.values())

# 2. pid is never missing
print("docs without pid:", sum(1 for d in docs if not d.get("pid")))

# 3. how often is each field missing? ("when present")
missing = {f: sum(1 for v in doc_store.values() if v[f] in (None, "", [])) for f in OUTPUT_FIELDS}
print(missing)

# 4. values are the original ones, not the processed ones
d0 = docs[0]
print(doc_store[d0["pid"]]["title"] == d0["title"])   # True: original text, not stemmed

docs without pid: 0
{'pid': 0, 'title': 0, 'description': 11150, 'brand': 2009, 'category': 0, 'sub_category': 0, 'product_details': 0, 'seller': 1643, 'out_of_stock': 0, 'selling_price': 2, 'discount': 855, 'actual_price': 777, 'average_rating': 2261, 'url': 0}
True


### 2.0 Setup and analysis decisions

Decisions taken for the whole analysis:

- **Which data.** The analysis uses `processed` (the output of Task 1), so the statistics describe exactly what will be indexed. Text statistics are reported **before** (raw words) and **after** pre-processing (stop words removed, stemmed).
- **Readable words for plots.** Frequency plots and word clouds use lowercase words *without stemming* (stop words and punctuation removed), because stems such as `friendli` are hard to read. Vocabulary statistics use the stemmed terms, because those are the terms the inverted index will contain.
- **Missing values** are never imputed. Each statistic is computed on the documents that have the field, and the amount of missing data is reported separately.
- **Skewed variables** (prices) are plotted on a log scale.
- **Rankings of brands / sellers by average value** only include those with at least `MIN_PRODUCTS` products, so that a brand with a single product does not top the ranking by chance.
- **Reproducibility.** The only sampling (named entity recognition) uses a fixed seed. All figures are also saved to the `figures/` folder, ready for the report.

In [ ]:
import os
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:          # running outside Jupyter
    display = print

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
MIN_PRODUCTS = 20            # minimum products for brand / seller average rankings
TOP_N = 10
SEED = 42

def finish(name):
    """Save the current figure to figures/<name>.png and show it."""
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

# `processed` is created in Task 1; reload it from disk if this section is run on its own
try:
    processed
except NameError:
    with open("processed_docs.json", encoding="utf-8") as f:
        processed = json.load(f)

df = pd.DataFrame(processed)

def is_missing(v):
    if v is None:
        return True
    if isinstance(v, float) and np.isnan(v):
        return True
    if isinstance(v, str):
        return v.strip() == ""
    if isinstance(v, (list, dict)):
        return len(v) == 0
    return False

def raw_words(text):
    """All lowercase alphanumeric words, nothing removed (used for 'before pre-processing' counts)."""
    return re.findall(r"[a-z0-9]+", text.lower()) if isinstance(text, str) else []

def readable_tokens(text):
    """Lowercase words without stop words or stemming (used for plots and word clouds)."""
    if not isinstance(text, str) or not text.strip():
        return []
    return drop_short(remove_stopwords(remove_punctuation(normalize(text)).split()))

# cleaned categorical columns: missing values become NaN
for col in ["brand", "seller", "category", "sub_category"]:
    df[col + "_clean"] = df[col].where(~df[col].map(is_missing))

print(df.shape)
df[["pid", "title", "brand", "category", "sub_category", "seller",
    "out_of_stock", "selling_price", "discount", "actual_price", "average_rating"]].head()

### 2.1 Dataset overview and missing values

How many documents are there, is `pid` really unique, and how complete is each field? Missing values matter for retrieval: a field with many gaps (for example the description) cannot be the only source of evidence for a document.

In [ ]:
print(f"documents: {len(df)}   unique pids: {df['pid'].nunique()}")
print(f"documents sharing their title with another one: {df['title'].duplicated(keep=False).sum()}")

missing_pct = pd.Series({f: df[f].map(is_missing).mean() * 100 for f in OUTPUT_FIELDS}).sort_values()
display(missing_pct.round(2).to_frame("missing %").sort_values("missing %", ascending=False))

ax = missing_pct.plot.barh(color="tab:blue")
ax.set_xlabel("% of documents with the field missing / empty")
ax.set_title("Missing values per field")
finish("missing_values")

### 2.2 Text statistics: length, sentences and vocabulary

Word counts per document, sentence length and vocabulary size for `title` and `description`. Descriptions that are missing are left out of the description statistics. Sentences are split on `. ! ?` (not when followed by a digit, so `3.5` is not split); this also splits `look.Designed`, where the crawled text lacks a space.

In [ ]:
title_len = df["title"].map(lambda t: len(raw_words(t)))
has_desc = ~df["description"].map(is_missing)
desc_len = df.loc[has_desc, "description"].map(lambda t: len(raw_words(t)))

print("Title length (words)");       display(title_len.describe().round(2).to_frame().T)
print("Description length (words)"); display(desc_len.describe().round(2).to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(title_len, bins=range(0, int(title_len.max()) + 2), color="tab:blue")
axes[0].set_title("Words per title"); axes[0].set_xlabel("words"); axes[0].set_ylabel("documents")
cap = desc_len.quantile(0.99)
axes[1].hist(desc_len.clip(upper=cap), bins=40, color="tab:orange")
axes[1].set_title(f"Words per description (clipped at the 99th percentile = {cap:.0f})")
axes[1].set_xlabel("words")
finish("word_count_distribution")

In [ ]:
def split_sentences(text):
    return [s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]

sent_per_desc = df.loc[has_desc, "description"].map(lambda t: len(split_sentences(t)))
sent_len = [len(raw_words(s)) for t in df.loc[has_desc, "description"] for s in split_sentences(t)]
avg_sent_len = float(np.mean(sent_len))

print(f"average sentences per description: {sent_per_desc.mean():.2f}")
print(f"average sentence length: {avg_sent_len:.2f} words (median {np.median(sent_len):.0f})")

plt.hist(np.clip(sent_len, 0, np.percentile(sent_len, 99)), bins=40, color="tab:green")
plt.title("Sentence length in descriptions (words)")
plt.xlabel("words per sentence"); plt.ylabel("sentences")
finish("sentence_length")

In [ ]:
rows = []
for name in ["title", "description"]:
    raw_tokens = [w for t in df[name] for w in raw_words(t)]
    terms = [w for ts in df[name + "_terms"] for w in ts]
    rows.append({"field": name,
                 "tokens before": len(raw_tokens), "vocabulary before": len(set(raw_tokens)),
                 "tokens after": len(terms),       "vocabulary after": len(set(terms))})
all_terms_flat = [w for ts in df["all_terms"] for w in ts]
rows.append({"field": "all_terms (merged)", "tokens before": np.nan, "vocabulary before": np.nan,
             "tokens after": len(all_terms_flat), "vocabulary after": len(set(all_terms_flat))})

vocab_table = pd.DataFrame(rows).set_index("field")
vocab_table["vocabulary reduction %"] = (1 - vocab_table["vocabulary after"] / vocab_table["vocabulary before"]) * 100
vocab_table["type/token ratio (after)"] = vocab_table["vocabulary after"] / vocab_table["tokens after"]
display(vocab_table.round(3))

### 2.3 Word frequencies, Zipf's law and word clouds

Most frequent words (readable, unstemmed) in titles and descriptions, the rank-frequency curve of the indexed terms, and word clouds. If `wordcloud` is not installed (`pip install wordcloud`) the clouds are replaced by bar charts.

In [ ]:
title_counter = Counter(w for t in df["title"] for w in readable_tokens(t))
desc_counter = Counter(w for t in df["description"] for w in readable_tokens(t))

def plot_top(counter, n, title, ax):
    words, counts = zip(*counter.most_common(n))
    ax.barh(words[::-1], counts[::-1], color="tab:blue")
    ax.set_title(title); ax.set_xlabel("occurrences")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plot_top(title_counter, 20, "Top 20 words in titles", axes[0])
plot_top(desc_counter, 20, "Top 20 words in descriptions", axes[1])
finish("top_words")

In [ ]:
term_counter = Counter(all_terms_flat)
freqs = np.array(sorted(term_counter.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)

hapax = int((freqs == 1).sum())
cum = np.cumsum(freqs) / freqs.sum()
print(f"indexed vocabulary: {len(freqs)} terms, {freqs.sum()} tokens")
print(f"terms appearing only once: {hapax} ({hapax / len(freqs) * 100:.1f}% of the vocabulary)")
print(f"terms needed to cover 80% of all tokens: {int(np.searchsorted(cum, 0.8)) + 1}")

plt.loglog(ranks, freqs, color="tab:red")
plt.title("Rank-frequency of indexed terms (log-log)")
plt.xlabel("rank"); plt.ylabel("frequency")
finish("zipf")

In [ ]:
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False

if HAS_WORDCLOUD:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (name, ctr) in zip(axes, [("Titles", title_counter), ("Descriptions", desc_counter)]):
        wc = WordCloud(width=800, height=500, background_color="white", max_words=100,
                       random_state=SEED).generate_from_frequencies(ctr)
        ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(f"Word cloud: {name}")
    finish("word_clouds")
else:
    print("wordcloud is not installed (pip install wordcloud): showing bar charts instead")
    fig, axes = plt.subplots(1, 2, figsize=(12, 8))
    plot_top(title_counter, 30, "Top 30 words in titles", axes[0])
    plot_top(desc_counter, 30, "Top 30 words in descriptions", axes[1])
    finish("word_clouds_fallback")

### 2.4 Categories, brands and sellers

Distribution of `category` and `sub_category`, and the concentration of products among brands and sellers. A very uneven distribution means brand or seller terms are very distinctive for a few documents and useless for the rest, which is relevant to the choice in 1.3.

In [ ]:
print("Products per category")
display(df["category_clean"].value_counts(dropna=False).to_frame("products"))

sub_counts = df["sub_category_clean"].value_counts()
print(f"{len(sub_counts)} sub-categories")
sub_counts.head(15)[::-1].plot.barh(color="tab:blue")
plt.title("Products per sub-category (top 15)"); plt.xlabel("products")
finish("sub_categories")

In [ ]:
def concentration(col, label):
    counts = df[col].value_counts()
    covered = counts.head(TOP_N).sum() / counts.sum() * 100
    print(f"{label}: {len(counts)} distinct values, {df[col].isna().sum()} products without one; "
          f"the top {TOP_N} cover {covered:.1f}% of the products that have one; "
          f"{(counts == 1).sum()} appear only once")
    return counts

brand_counts = concentration("brand_clean", "brands")
seller_counts = concentration("seller_clean", "sellers")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
brand_counts.head(15)[::-1].plot.barh(ax=axes[0], color="tab:blue"); axes[0].set_title("Top 15 brands by number of products")
seller_counts.head(15)[::-1].plot.barh(ax=axes[1], color="tab:orange"); axes[1].set_title("Top 15 sellers by number of products")
for ax in axes:
    ax.set_xlabel("products")
finish("top_brands_sellers")

### 2.5 Stock, prices, discounts and ratings

Distribution of the non-textual fields from 1.4, a consistency check between `discount`, `actual_price` and `selling_price`, and the correlations between them.

In [ ]:
oos = df["out_of_stock"].map(lambda v: np.nan if is_missing(v) else float(v))
oos_counts = df["out_of_stock"].value_counts(dropna=False)
display(oos_counts.to_frame("products"))
print(f"out of stock: {np.nanmean(oos) * 100:.2f}% of the products with a value")

oos_by_sub = (df.assign(oos=oos).groupby("sub_category_clean")["oos"]
                .agg(["mean", "count"]))
oos_by_sub = oos_by_sub[oos_by_sub["count"] >= MIN_PRODUCTS].sort_values("mean", ascending=False)
if len(oos_by_sub):
    (oos_by_sub["mean"].head(10)[::-1] * 100).plot.barh(color="tab:red")
    plt.title(f"% out of stock by sub-category (at least {MIN_PRODUCTS} products)")
    plt.xlabel("% out of stock")
    finish("out_of_stock_by_subcategory")

In [ ]:
num_cols = ["selling_price", "actual_price", "discount", "average_rating"]
display(df[num_cols].describe().round(2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, color in [("selling_price", "tab:blue"), ("actual_price", "tab:orange")]:
    vals = df[col][df[col] > 0].dropna()
    axes[0, 0].hist(np.log10(vals), bins=40, alpha=0.6, label=col, color=color)
axes[0, 0].set_title("Prices (log10 scale)"); axes[0, 0].set_xlabel("log10(price)"); axes[0, 0].legend()
axes[0, 1].hist(df["discount"].dropna(), bins=40, color="tab:green")
axes[0, 1].set_title("Discount (%)"); axes[0, 1].set_xlabel("% off")
axes[1, 0].hist(df["average_rating"].dropna(), bins=np.arange(0.5, 5.6, 0.25), color="tab:purple")
axes[1, 0].set_title("Average rating"); axes[1, 0].set_xlabel("rating")
axes[1, 1].scatter(df["actual_price"], df["selling_price"], s=3, alpha=0.3)
axes[1, 1].set_xscale("log"); axes[1, 1].set_yscale("log")
axes[1, 1].set_title("Selling vs actual price"); axes[1, 1].set_xlabel("actual price"); axes[1, 1].set_ylabel("selling price")
finish("numeric_distributions")

In [ ]:
# consistency: discount should be roughly (actual - selling) / actual
computed = ((df["actual_price"] - df["selling_price"]) / df["actual_price"] * 100).replace([np.inf, -np.inf], np.nan)
gap = (computed - df["discount"]).abs().dropna()
print(f"products where discount, actual and selling price are all present: {len(gap)}")
print(f"  discount within 1 percentage point of the computed value: {(gap <= 1).mean() * 100:.1f}%")
print(f"  selling price higher than actual price: {(df['selling_price'] > df['actual_price']).sum()} products")

corr = df[num_cols].assign(out_of_stock=oos).corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.grid(False); fig.colorbar(im); ax.set_title("Correlation between numeric fields")
finish("correlations")

### 2.6 Rankings of products, brands and sellers

Products ranked by rating, price and discount, and brands and sellers ranked by number of products and by average rating (only those with at least `MIN_PRODUCTS` products). Ratings carry no number of reviews in this dataset, so a high rating can come from very few reviews.

In [ ]:
def top_products(col, n=TOP_N, ascending=False, min_value=None):
    sub = df.dropna(subset=[col])
    if min_value is not None:
        sub = sub[sub[col] > min_value]
    return sub.sort_values(col, ascending=ascending).head(n)[["pid", "title", "brand", "seller", col]]

max_rating = df["average_rating"].max()
print(f"products with a rating: {df['average_rating'].notna().sum()}; "
      f"{(df['average_rating'] == max_rating).sum()} of them share the maximum rating ({max_rating})")

print("\nTop products by rating (ties are in arbitrary order)")
display(top_products("average_rating"))
print("Most expensive products (selling price)")
display(top_products("selling_price"))
print("Cheapest products (selling price > 0)")
display(top_products("selling_price", ascending=True, min_value=0))
print("Highest discounts")
display(top_products("discount"))

In [ ]:
def group_stats(col):
    g = df.groupby(col).agg(products=("pid", "count"), mean_rating=("average_rating", "mean"),
                            mean_price=("selling_price", "mean"), mean_discount=("discount", "mean"),
                            pct_out_of_stock=("out_of_stock", lambda s: s.map(lambda v: np.nan if is_missing(v) else float(v)).mean() * 100))
    return g.round(2)

brand_stats = group_stats("brand_clean")
seller_stats = group_stats("seller_clean")

for name, stats in [("brands", brand_stats), ("sellers", seller_stats)]:
    big = stats[stats["products"] >= MIN_PRODUCTS]
    print(f"\nTop {TOP_N} {name} by number of products"); display(stats.sort_values("products", ascending=False).head(TOP_N))
    print(f"Top {TOP_N} {name} by mean rating (at least {MIN_PRODUCTS} products; {len(big)} qualify)")
    display(big.sort_values("mean_rating", ascending=False).head(TOP_N))
    print(f"Top {TOP_N} {name} by mean discount (at least {MIN_PRODUCTS} products)")
    display(big.sort_values("mean_discount", ascending=False).head(TOP_N))

In [ ]:
top_subs = df["sub_category_clean"].value_counts().head(8).index
groups = [df.loc[(df["sub_category_clean"] == s) & (df["selling_price"] > 0), "selling_price"].dropna() for s in top_subs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(groups, showfliers=False)
axes[0].set_xticks(range(1, len(top_subs) + 1)); axes[0].set_xticklabels(list(top_subs))
axes[0].set_title("Selling price by sub-category (8 largest, no outliers)")
axes[0].tick_params(axis="x", rotation=45)
df.groupby("sub_category_clean")["discount"].mean().loc[top_subs].plot.bar(ax=axes[1], color="tab:green")
axes[1].set_title("Mean discount (%) by sub-category"); axes[1].tick_params(axis="x", rotation=45)
finish("price_discount_by_subcategory")

### 2.7 Contents of `product_details`

`product_details` is a list of key/value pairs that differ per product. Which attributes are common? Attributes present in almost every product (and with few distinct values) say little about a specific product, while rarer ones are more distinctive. This supports the treatment of `details_terms` in 1.3.

In [ ]:
flat_details = [flatten_details(d) for d in df["product_details"]]
n_attrs = pd.Series([len(fd) for fd in flat_details])
key_counter = Counter(k for fd in flat_details for k in fd)

print(f"attributes per product: mean {n_attrs.mean():.1f}, median {n_attrs.median():.0f}, max {n_attrs.max()}; "
      f"products without any: {(n_attrs == 0).sum()}")
print(f"{len(key_counter)} distinct attribute names\n")

key_table = pd.DataFrame({"products": pd.Series(key_counter)}).sort_values("products", ascending=False)
key_table["% of products"] = (key_table["products"] / len(df) * 100).round(1)
key_table["distinct values"] = [len({fd[k] for fd in flat_details if k in fd}) for k in key_table.index]
display(key_table.head(15))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key in zip(axes, ["Color", "Fabric"]):
    values = Counter(fd[key] for fd in flat_details if key in fd)
    if values:
        plot_top(values, 12, f"Most common values of '{key}'", ax)
    else:
        ax.set_title(f"No '{key}' attribute found")
finish("product_details_values")

### 2.8 The two evaluation queries

Following the hint of the statement, this section looks at how the terms of the two validation queries are spread over the fields (using the same pre-processing as the documents). It shows in which fields the evidence for each query term can be found, and how many documents contain all of the query terms.

In [ ]:
QUERIES = {"query_1": "women full sleeve sweatshirt cotton", "query_2": "men slim jeans blue"}
FIELD_COLS = ["title_terms", "description_terms", "details_terms", "brand_terms", "category_terms", "all_terms"]
term_sets = {c: df[c].map(set) for c in FIELD_COLS}

rows = []
for qname, q in QUERIES.items():
    qterms = build_terms(q)
    print(f"{qname}: '{q}' -> {qterms}")
    for t in qterms:
        rows.append({"query": qname, "term": t, **{c.replace("_terms", ""): int(term_sets[c].map(lambda s: t in s).sum())
                                                   for c in FIELD_COLS}})
    n_all = int(term_sets["all_terms"].map(lambda s: all(t in s for t in qterms)).sum())
    n_title = int(term_sets["title_terms"].map(lambda s: all(t in s for t in qterms)).sum())
    print(f"   documents containing every query term: {n_all} (anywhere), {n_title} (all in the title)")

print("\nNumber of documents containing each query term, per field")
display(pd.DataFrame(rows).set_index(["query", "term"]))

### 2.9 Named entity recognition

Named entities (brands, places, ...) found in a random sample of descriptions with spaCy (`en_core_web_sm`). Titles are not used because they are short noun phrases without a sentence context. The entity recogniser is trained on general text, so fashion vocabulary may be mislabelled; the results are only indicative. This cell needs spaCy and its model, and is skipped otherwise.

In [ ]:
N_NER_SAMPLE = 2000
try:
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
    HAS_SPACY = True
except (ImportError, OSError):
    HAS_SPACY = False
    print("spaCy or the model en_core_web_sm is not available. Install them with:\n"
          "  pip install spacy\n  python -m spacy download en_core_web_sm")

if HAS_SPACY:
    sample = df.loc[has_desc, "description"].sample(min(N_NER_SAMPLE, int(has_desc.sum())), random_state=SEED)
    label_counter, entity_counter = Counter(), Counter()
    for doc_ in nlp.pipe(sample.tolist(), batch_size=64):
        for ent in doc_.ents:
            label_counter[ent.label_] += 1
            entity_counter[(ent.label_, ent.text.strip())] += 1

    print(f"{sum(label_counter.values())} entities found in {len(sample)} descriptions")
    labels, counts = zip(*label_counter.most_common())
    plt.bar(labels, counts, color="tab:blue"); plt.xticks(rotation=45)
    plt.title("Named entity types in descriptions (sample)"); plt.ylabel("entities")
    finish("ner_labels")

    for label in ["ORG", "GPE", "NORP", "PERSON", "PRODUCT"]:
        top = [(e, c) for (l, e), c in entity_counter.most_common() if l == label][:8]
        if top:
            print(f"{label}: {top}")

### 2.10 Summary of the main figures

The numbers below are computed from the cells above, so they can be copied into the report. They are printed rather than written here, so they always match the data that was actually loaded.

In [ ]:
print(f"documents: {len(df)}")
print(f"missing: description {missing_pct['description']:.1f}%, brand {missing_pct['brand']:.1f}%, "
      f"seller {missing_pct['seller']:.1f}%, rating {missing_pct['average_rating']:.1f}%, "
      f"discount {missing_pct['discount']:.1f}%")
print(f"title: {title_len.mean():.1f} words on average; description: {desc_len.mean():.1f} words on average, "
      f"{avg_sent_len:.1f} words per sentence")
print(f"vocabulary (title / description / merged, after pre-processing): "
      f"{vocab_table.loc['title', 'vocabulary after']:.0f} / {vocab_table.loc['description', 'vocabulary after']:.0f} / "
      f"{vocab_table.loc['all_terms (merged)', 'vocabulary after']:.0f}")
print(f"{len(brand_counts)} brands, {len(seller_counts)} sellers, {df['sub_category_clean'].nunique()} sub-categories")
print(f"out of stock: {np.nanmean(oos) * 100:.2f}%")
print(f"selling price: median {df['selling_price'].median():.0f}; discount: median {df['discount'].median():.0f}%; "
      f"rating: median {df['average_rating'].median():.1f}")
print(f"figures saved in ./{FIG_DIR}/: {len(os.listdir(FIG_DIR))} files")